## Multicollinearity

### Problem: Predict Student Exam Score

- We want to predict a student’s final exam score based on:
  - Hours_Studied (main study time)
  - Sleep_Hours (rest)
  - Homework_Completed (highly correlated with Hours_Studied)

 **We’ll see how multicollinearity affects regression.**

### Step 1: Create the synthetic Dataset

**Explanation of below code:**
- Hours_Studied and Sleep_Hours → main predictors
- Homework_Completed → highly correlated with Hours_Studied
- Score → target we want to predict
- Noise (np.random.normal) → makes it more like real-world data

In [1]:
import pandas as pd
import numpy as np

np.random.seed(1)

# Features
hours_studied = np.random.randint(1, 10, 100)
sleep_hours = np.random.randint(4, 10, 100)

# Create a collinear feature: Homework_Completed
# Most students who study more complete more homework
homework_completed = hours_studied + np.random.normal(0, 1, 100)

# Target (exam score)
score = 5*hours_studied + 3*sleep_hours + np.random.normal(0, 2, 100)

# Create DataFrame
df = pd.DataFrame({
    "Hours_Studied": hours_studied,
    "Sleep_Hours": sleep_hours,
    "Homework_Completed": homework_completed,
    "Score": score
})

df.head()

,Hours_Studied,Sleep_Hours,Homework_Completed,Score
0,6,6,6.846161,50.228766
1,9,8,8.140484,67.826952
2,6,8,6.350546,51.526293
3,1,6,-0.312283,24.751678
4,1,6,0.961304,24.246724


### Step 2: Train Linear Regression (without Collinearity)

In [2]:
from sklearn.linear_model import LinearRegression

# Features:  only Hours_Studied and Sleep_Hours
X = df[["Hours_Studied", "Sleep_Hours"]]
y = df["Score"]

model = LinearRegression()
model.fit(X, y)

print("Intercept:", model.intercept_)
print("Coefficients:", model.coef_)

Intercept: -0.9575992876384305
Coefficients: [4.97833723 3.18345842]


**Interpretation**

- Hours_Studied ≈ 4.98 → close to the “true” 5
- Sleep_Hours ≈ 3.18 → close to the “true” 3
- Coefficients are stable and intuitive
- Model is telling us exactly how each feature contributes to the score

### Step 3: Add the collinear feature (Homework_Completed)

In [4]:
# Include Homework_Completed now
X_collinear = df[["Hours_Studied", "Homework_Completed", "Sleep_Hours"]]

model_collinear = LinearRegression()
model_collinear.fit(X_collinear, y)

print("Intercept:", model_collinear.intercept_)
print("Coefficients(collinearity):", model_collinear.coef_)

Intercept: -1.0299977107394724
Coefficients(collinearity): [4.8005962  0.17360494 3.19480586]


**Interpretation**
- Hours_Studied and Homework_Completed are highly correlated
- The model is uncertain which feature “explains” the target
- As a result, the weight (coefficient) of Homework_Completed is small and unstable
- The sum of the two weights (Hours_Studied + Homework_Completed) is roughly same as before → prediction is still good

In [5]:
df.corr()

,Hours_Studied,Sleep_Hours,Homework_Completed,Score
Hours_Studied,1.000000,0.226381,0.939594,0.927453
Sleep_Hours,0.226381,1.000000,0.173987,0.555348
Homework_Completed,0.939594,0.173987,1.000000,0.861009
Score,0.927453,0.555348,0.861009,1.000000
